# 07. 실측 유전자 섭동 반응 예측

05의 Geneformer 가상 삭제는 embedding 변화 탐색입니다. 이 노트북은 실제 control/treated 발현량으로 평가한 별도 task를 읽습니다. 기본 동작은 저장된 결과 읽기이며 GEARS 전체 학습·원본 다운로드를 시작하지 않습니다.


In [1]:
from pathlib import Path
import json, sys, subprocess
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
REPORT = ROOT / 'reports/research_phase2'


## 데이터와 평가 단위

Norman은 공식 전처리 K562 CRISPRa 데이터와 GEARS simulation seed 1 split입니다. HepG2는 GSE264667의 day-7 CRISPRi 데이터입니다. 서로 다른 세포주·개입이므로 하나의 점수로 합치지 않습니다. HepG2의 dual sgRNA는 동일 유전자 표적 설계이며 두 유전자 조합 KO로 취급하지 않습니다. 모든 guide/transcript 변형을 target 유전자별로 묶어 holdout했습니다.

HepG2의 기술 배치(gem_group)마다 non-targeting control을 제공합니다. biological replicate는 unknown으로 기록했습니다. 출력 유전자 2,000개는 학습 target 세포와 제공된 control만 이용해 고릅니다. 평균 발현량과 변화량은 log1p(10k-normalized counts) 공간에서 비교합니다. 성인 HCC 환자 약효나 임상 반응으로 해석하지 않습니다.


In [2]:
contract = json.loads((REPORT / 'hepg2_data_contract/contract.json').read_text())
print('eligible target counts:', contract['target_split_counts'])
print('control policy:', contract['control_policy'])
display(pd.read_csv(REPORT / 'hepg2_data_contract/batch_control_support.tsv', sep='\t').head())


eligible target counts: {'train': 1714, 'test': 214, 'val': 214}
control policy: Non-targeting controls from the same gem_group are available for training and evaluation; held-out treated responses never enter training


,batch,n_non_targeting_controls,biological_replicate
0,3,116,unknown
1,12,91,unknown
2,53,92,unknown
3,39,91,unknown
4,24,75,unknown


## 간단한 예측기와 실제 측정값 비교

B00은 변화가 없다는 matched control 평균, B01은 학습 intervention의 평균 변화, B02는 외부 GO membership에서 학습 반응 PCA 계수를 예측하는 ridge입니다. B02의 PCA는 학습 response만 사용하고 alpha는 validation MSE로 선택합니다. 새로운 target의 실측 response를 intervention feature로 사용하지 않습니다.

MSE와 Δ correlation은 intervention별 평균을 동일 가중치로 합칩니다. No-change의 Δ correlation은 정의되지 않으므로 NaN입니다. 원 발현량 correlation이 높아도 변화 예측이 좋다는 뜻은 아닙니다. `observed_top20_delta`는 평가 시 실측 평균 변화의 절댓값 상위 20개를 보는 보조 지표이며 predictor 입력·선택에는 사용하지 않았습니다.


In [3]:
for name in ['B00_B02_norman_simulation_seed1_v2', 'B00_B02_hepg2_unseen_target_seed42_v1']:
    print(name)
    display(pd.read_csv(REPORT / 'runs' / name / 'metrics.tsv', sep='\t'))


B00_B02_norman_simulation_seed1_v2


,model_id,n_cells,mse,mae,pearson_expression,pearson_delta,observed_mean_delta_l2,mse_observed_top20_delta,sign_accuracy_observed_top20_delta
0,B00_no_change,268.728972,0.006831,0.022436,0.979402,NaN,5.391163,0.642751,0.000000
1,B01_train_mean,268.728972,0.004989,0.019206,0.984842,0.578628,5.391163,0.475942,0.870561
2,B02_GO_PCA_ridge,268.728972,0.002839,0.015579,0.991331,0.703796,5.391163,0.235853,0.891589


B00_B02_hepg2_unseen_target_seed42_v1


,model_id,n_cells,mse,mae,pearson_expression,pearson_delta,observed_mean_delta_l2,mse_observed_top20_delta,sign_accuracy_observed_top20_delta
0,B00_no_change,65.392523,0.019060,0.093090,0.956944,NaN,5.533477,0.304628,0.000000
1,B01_train_mean,65.392523,0.016177,0.089355,0.962942,0.324815,5.533477,0.226941,0.734813
2,B02_GO_PCA_ridge,65.392523,0.012566,0.080464,0.971014,0.394152,5.533477,0.152947,0.784813


## GEARS 실행 상태

GEARS는 실제 control/treated graph를 학습하는 별도 모델입니다. 아래는 완료된 run.json이 있을 때만 점수를 표시합니다. 공식 Norman은 원 DE 처리와 simulation split을 유지하는 재현이며, HepG2 strict 정책과 별도 관리합니다. 학습 중인 작업을 완료로 표시하지 않습니다.


In [4]:
path = REPORT / 'runs/R01_norman_simulation_seed1_v2/run.json'
if path.exists():
    gears = json.loads(path.read_text())
    print(gears['status'], gears['metrics'])
else:
    print('완료된 GEARS run.json이 아직 없습니다. 이 노트북은 학습을 시작하지 않습니다.')


executed {'mse': '0.005285478', 'mse_de': '0.18092044', 'pearson': 0.9850512477856492, 'pearson_de': 0.8665455014664131}


## 선택적인 CPU 기준선 재실행

이미 준비된 HepG2 task와 GO prior를 사용합니다. 원본 5GB 파일 변환이나 전체 GEARS 학습을 다시 수행하지 않습니다. 기존 결과를 보존하기 위해 새 run ID를 만듭니다.


In [5]:
RUN_BASELINES = False
if RUN_BASELINES:
    from datetime import datetime, timezone
    run_id = 'tutorial07_hepg2_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
    subprocess.run([sys.executable, str(ROOT / 'scripts/research/run_response_baselines.py'), '--task', 'hepg2', '--run-id', run_id], cwd=ROOT, check=True)
    display(pd.read_csv(REPORT / 'runs' / run_id / 'metrics.tsv', sep='\t'))
else:
    print('저장 결과 읽기 모드')


저장 결과 읽기 모드


## 통일된 조건별 RMSE와 retrieval 평가

원 모델을 재학습하지 않고 실제 저장 예측에서 계산합니다. Top20은 관측 평균 변화량의 절댓값으로 정한 evaluator-only 목록이며 통계적 FDR DE가 아닙니다. Constant prediction의 Pearson/P@20은 undefined로 둡니다.


In [6]:
for task in ['norman','hepg2']:
    print('task:',task)
    display(pd.read_csv(REPORT/f'response_evaluation_{task}/condition_macro_metrics.tsv',sep='	'))
    display(pd.read_csv(REPORT/f'response_evaluation_{task}/paired_intervention_bootstrap.tsv',sep='	'))
print('Native scGPT pretrained/random 및 matched GEARS는 별도3303-gene/191-target 공통 panel이며 full HepG2214-target 점수와 합치지 않습니다.')
path=REPORT/'response_evaluation_native_common/condition_macro_metrics.tsv'
if path.exists(): display(pd.read_csv(path,sep='	'))
else: print('공통 panel 비교는 아직 실행 중이며 완료된 성능으로 표시하지 않습니다.')


task: norman


,model_id,delta_rmse,delta_mae,pearson_delta,top20_response_retrieval_average_precision,top20_response_retrieval_precision_at20,delta_rmse_excluding_direct_target,sign_accuracy_observed_top20_delta
0,B00_no_change,0.075902,0.022436,NaN,0.003964,NaN,0.074826,0.000000
1,B01_train_mean,0.064405,0.019206,0.578628,0.309311,0.346729,0.063209,0.870561
2,B02_GO_PCA_ridge,0.050039,0.015579,0.703796,0.469553,0.462150,0.049136,0.891589
3,B03_GEARS,0.069212,0.030618,0.538014,0.305861,0.342523,0.068385,0.894860


,model_id,delta_RMSE_gain_vs_no_change,lower95,upper95,n_interventions
0,B00_no_change,0.000000,0.000000,0.000000,107
1,B01_train_mean,0.011497,0.009826,0.013053,107
2,B02_GO_PCA_ridge,0.025863,0.020849,0.031209,107
3,B03_GEARS,0.006690,0.003277,0.010122,107


task: hepg2


,model_id,delta_rmse,delta_mae,pearson_delta,top20_response_retrieval_average_precision,top20_response_retrieval_precision_at20,delta_rmse_excluding_direct_target,sign_accuracy_observed_top20_delta
0,B00_no_change,0.123732,0.093090,NaN,0.010000,NaN,0.123422,0.000000
1,B01_train_mean,0.118486,0.089355,0.324815,0.167027,0.190888,0.118190,0.734813
2,B02_GO_PCA_ridge,0.105837,0.080464,0.394152,0.232237,0.247430,0.105485,0.784813
3,B03_GEARS,0.130913,0.100603,0.221344,0.135514,0.158178,0.130637,0.695794


,model_id,delta_RMSE_gain_vs_no_change,lower95,upper95,n_interventions
0,B00_no_change,0.000000,0.000000,0.000000,214
1,B01_train_mean,0.005247,0.002363,0.008133,214
2,B02_GO_PCA_ridge,0.017895,0.012073,0.023921,214
3,B03_GEARS,-0.007180,-0.012227,-0.001908,214


Native scGPT pretrained/random 및 matched GEARS는 별도3303-gene/191-target 공통 panel이며 full HepG2214-target 점수와 합치지 않습니다.


,model_id,delta_rmse,delta_mae,pearson_delta,top20_response_retrieval_average_precision,top20_response_retrieval_precision_at20,delta_rmse_excluding_direct_target,sign_accuracy_observed_top20_delta
0,B00_no_change,0.126950,0.095465,NaN,0.010116,NaN,0.126599,0.000000
1,B01_train_mean,0.120503,0.090863,0.347284,0.166179,0.186387,0.120172,0.755236
2,B02_GO_PCA_ridge,0.110730,0.083900,0.394165,0.215437,0.229581,0.110359,0.781414
3,B03_GEARS,0.137319,0.105232,0.162119,0.114678,0.139005,0.137027,0.634031
4,B04_scGPT_pretrained,0.123941,0.093652,0.300043,0.134198,0.153403,0.123936,0.721466
5,B04_scGPT_random,0.122679,0.092923,0.330183,0.163927,0.183770,0.122674,0.754188


## 실제 intervention 한 개의 관측값과 예측

고정 test 목록의 첫 target을 사용합니다. 잘 맞는 사례를 골라 보여주지 않습니다. Gene 표시 순서는 관측 평균 변화량의 절댓값 Top20이며 evaluator-only입니다. 각 값은 서로 다른 세포 집단의 평균으로, paired-cell 전후 관측이 아닙니다.


In [7]:
example=json.loads((REPORT/'response_evaluation_hepg2/example_selection.json').read_text())
print('target:',example['target'],'observed treated cells:',example['n_treated_cells'])
print(example['target_selection'])
display(pd.read_csv(REPORT/'response_evaluation_hepg2/example_lexfirst_target.tsv',sep='\t'))
display(pd.read_csv(REPORT/'response_evaluation_hepg2/by_observed_guide_assignments.tsv',sep='\t'))
print('Guide assignment는 독립 biological replicate를 뜻하지 않습니다.')


target: AAR2 observed treated cells: 38
Lexicographically first fixed test target, no performance-based selection


,gene_symbol,control_mean,observed_treated_mean,observed_mean_delta,B00_no_change_predicted_mean,B00_no_change_predicted_delta,B01_train_mean_predicted_mean,B01_train_mean_predicted_delta,B02_GO_PCA_ridge_predicted_mean,B02_GO_PCA_ridge_predicted_delta,B03_GEARS_predicted_mean,B03_GEARS_predicted_delta
0,PFDN4,0.819378,0.079616,-0.739762,0.819378,0.0,0.819325,-0.000053,0.852807,0.033429,0.909744,0.090366
1,ESF1,0.748054,0.248869,-0.499184,0.748054,0.0,0.783834,0.035780,0.784728,0.036674,0.837783,0.089729
2,CPED1,0.464775,0.120532,-0.344243,0.464775,0.0,0.434951,-0.029825,0.465820,0.001045,0.440733,-0.024042
3,MALAT1,3.399141,3.095695,-0.303447,3.399141,0.0,3.644860,0.245719,3.554270,0.155129,3.484529,0.085388
4,TUBA1B,2.535916,2.819101,0.283185,2.535916,0.0,2.154143,-0.381773,2.299233,-0.236683,2.255905,-0.280011
5,ID3,1.709987,1.433816,-0.276171,1.709987,0.0,1.649551,-0.060436,1.689623,-0.020364,1.591479,-0.118508
6,JTB,1.003879,1.249603,0.245724,1.003879,0.0,0.976947,-0.026932,0.962293,-0.041586,0.896282,-0.107596
7,ANKRD1,0.514124,0.276129,-0.237994,0.514124,0.0,0.534959,0.020835,0.560324,0.046200,0.607902,0.093778
8,SLC2A1,0.644353,0.881719,0.237366,0.644353,0.0,0.616465,-0.027888,0.730116,0.085762,0.890988,0.246635
9,TRAPPC2L,0.759735,0.996570,0.236835,0.759735,0.0,0.721448,-0.038288,0.722231,-0.037504,0.732696,-0.027039


,model_id,guide_assignment_stratum,delta_rmse,delta_mae,pearson_delta,top20_response_retrieval_average_precision,top20_response_retrieval_precision_at20,delta_rmse_excluding_direct_target,sign_accuracy_observed_top20_delta
0,B00_no_change,multiple_assignments,0.079792,0.059999,NaN,0.010000,NaN,0.079187,0.000000
1,B00_no_change,one_assignment,0.125218,0.094209,NaN,0.010000,NaN,0.124918,0.000000
2,B01_train_mean,multiple_assignments,0.082938,0.061829,0.286639,0.153810,0.185714,0.082251,0.671429
3,B01_train_mean,one_assignment,0.119688,0.090286,0.326105,0.167474,0.191063,0.119406,0.736957
4,B02_GO_PCA_ridge,multiple_assignments,0.072063,0.054450,0.337681,0.143704,0.192857,0.071335,0.764286
5,B02_GO_PCA_ridge,one_assignment,0.106980,0.081344,0.396062,0.235231,0.249275,0.106640,0.785507
6,B03_GEARS,multiple_assignments,0.097413,0.073916,0.130467,0.064898,0.107143,0.096842,0.657143
7,B03_GEARS,one_assignment,0.132045,0.101505,0.224417,0.137902,0.159903,0.131780,0.697101


Guide assignment는 독립 biological replicate를 뜻하지 않습니다.


In [8]:
REPLAY_ONE_SAVED_BASELINE = False
if REPLAY_ONE_SAVED_BASELINE:
    from datetime import datetime, timezone
    replay_dir=Path('/mnt/S1/sdata/pubdata/Virtual_Cell/gssong/vcell_liver/research_phase2/probes')/('tutorial07_replay_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S'))
    subprocess.run([sys.executable,str(ROOT/'scripts/research/replay_response_baseline.py'),'--task','hepg2','--output',str(replay_dir)],cwd=ROOT,check=True)
    print(json.loads((replay_dir/'replay.json').read_text()))
    display(pd.read_csv(replay_dir/'example.tsv',sep='\t'))
else:
    print('저장 결과 읽기 모드: 새 fitting이나 inference를 실행하지 않았습니다.')


저장 결과 읽기 모드: 새 fitting이나 inference를 실행하지 않았습니다.


## Native 대조의 평가 범위

주 평가는191targets/1977genes이며3303개 전체 shared gene 평가는 별도 보조 표입니다. Native GEARS/pretrained/random은 같은 실제 training cells와 fixed matched control cell IDs를 사용합니다. 통계적 mean 기준선은 같은 제공 control pool에서 조건별 평균을 계산하므로 입력 aggregation 방식은 다릅니다. 사전학습 효과의 직접 대조는 동일 구조·입력·optimizer의 pretrained vs random입니다.


In [9]:
population=json.loads((REPORT/'official_norman_population/audit.json').read_text())
print('Norman source/eligible/test cells:',population['n_source_cells'],population['n_official_GO_eligible_cells'],population['n_test_cells'])
display(pd.read_csv(REPORT/'response_evaluation_native_common/paired_model_comparisons.tsv',sep='\t'))
print('Secondary all3303 genes; do not merge with primary1977 or full HepG22000:')
display(pd.read_csv(REPORT/'response_evaluation_native_common/all3303_genes_condition_macro_metrics.tsv',sep='\t'))


Norman source/eligible/test cells: 91205 89357 28754


,reference,candidate,delta_RMSE_gain,lower95,upper95,n_interventions
0,B02_GO_PCA_ridge,B00_no_change,-0.016220,-0.021432,-0.011065,191
1,B02_GO_PCA_ridge,B01_train_mean,-0.009773,-0.012845,-0.006828,191
2,B02_GO_PCA_ridge,B03_GEARS,-0.026589,-0.028953,-0.024273,191
3,B02_GO_PCA_ridge,B04_scGPT_pretrained,-0.013211,-0.015560,-0.010947,191
4,B02_GO_PCA_ridge,B04_scGPT_random,-0.011948,-0.015034,-0.008970,191
5,B04_scGPT_random,B04_scGPT_pretrained,-0.001262,-0.003280,0.000856,191


Secondary all3303 genes; do not merge with primary1977 or full HepG22000:


,model_id,delta_rmse,delta_mae,pearson_delta,top20_response_retrieval_average_precision,top20_response_retrieval_precision_at20,delta_rmse_excluding_direct_target,sign_accuracy_observed_top20_delta
0,B00_no_change,0.106535,0.075928,NaN,0.006055,NaN,0.105972,0.000000
1,B01_train_mean,0.101268,0.072414,0.337661,0.161434,0.183508,0.100731,0.751309
2,B02_GO_PCA_ridge,0.093558,0.067508,0.385485,0.210551,0.226963,0.092977,0.779319
3,B03_GEARS,0.117405,0.086489,0.152591,0.109543,0.135340,0.116960,0.634031
4,B04_scGPT_pretrained,0.104783,0.075155,0.282753,0.136222,0.156545,0.104748,0.716230
5,B04_scGPT_random,0.103057,0.074336,0.323583,0.165683,0.187958,0.103025,0.754712


## V3: full-control 평균과 sampled matched controls

기존 모델·선택·예측을 보존한 감도 분석입니다. Simple predictor의 control 입력을 바꾸는 경우와 fixed prediction의 평가 reference만 바꾸는 경우를 구분합니다. 후자는 RMSE를 바꾸지 않지만 Delta Pearson은 공통 control noise 때문에 달라질 수 있습니다. 20seed 변동은 Monte Carlo 감도이며 biological replicate CI가 아닙니다. Saved scGPT 두 모델의 AAR2 38cells 추론은 점수와 무관하게 고정한 첫 target 하나의 사례입니다. 전체191target 비교로 일반화하지 않습니다.


In [10]:
display(pd.read_csv(REPORT/'control_sensitivity/aggregation_metrics.tsv',sep='\t'))
display(pd.read_csv(REPORT/'control_sensitivity/simple_predictor_resampling.tsv',sep='\t').groupby('model_id').mean(numeric_only=True))
probe=json.loads((REPORT/'control_sensitivity/native_first_target.json').read_text())
print('target:',probe['target'],'treated cells:',probe['n_treated_cells'],'new training:',probe['new_training'])
print('source runs:',[row['source_run_id'] for row in probe['evidence']])
display(pd.read_csv(REPORT/'control_sensitivity/native_first_target_metrics.tsv',sep='\t')[['initialization','input_control_policy','target','delta_rmse','pearson_delta']])
print('선택: 원 validation primary1977-gene intervention-mean MSE. 보고: test condition RMSE/Delta Pearson. Top20 recovery는 FDR DEG 검증이 아닙니다.')


,model_id,comparison,mean_intervention_RMSE,mean_intervention_delta_Pearson
0,B00_no_change,original_predictions_full_control_evaluator,0.126950,NaN
1,B00_no_change,fixed_predictions_recorded_sampled_control_eva...,0.126950,0.494743
2,B00_no_change,same_fitted_delta_with_recorded_sampled_contro...,0.145649,NaN
3,B01_train_mean,original_predictions_full_control_evaluator,0.120503,0.347284
4,B01_train_mean,fixed_predictions_recorded_sampled_control_eva...,0.120503,0.590815
5,B01_train_mean,same_fitted_delta_with_recorded_sampled_contro...,0.139054,0.308902
6,B02_GO_PCA_ridge,original_predictions_full_control_evaluator,0.110730,0.394165
7,B02_GO_PCA_ridge,fixed_predictions_recorded_sampled_control_eva...,0.110730,0.641542
8,B02_GO_PCA_ridge,same_fitted_delta_with_recorded_sampled_contro...,0.130501,0.353641
9,B03_GEARS,original_predictions_full_control_evaluator,0.137319,0.162119


,seed,mean_intervention_RMSE,mean_intervention_delta_Pearson
model_id,,,
B00_no_change,1009.5,0.145743,NaN
B01_train_mean,1009.5,0.138877,0.314039
B02_GO_PCA_ridge,1009.5,0.130067,0.356441


target: AAR2 treated cells: 38 new training: False
source runs: ['B04_pretrained_native_common_seed42_v1', 'B04_random_native_common_seed42_v1']


,initialization,input_control_policy,target,delta_rmse,pearson_delta
0,pretrained,recorded_sampled,AAR2,0.108824,-0.013833
1,pretrained,full_same_batch_mean,AAR2,0.108399,-0.018173
2,random,recorded_sampled,AAR2,0.104953,-0.002916
3,random,full_same_batch_mean,AAR2,0.105519,-0.000760


선택: 원 validation primary1977-gene intervention-mean MSE. 보고: test condition RMSE/Delta Pearson. Top20 recovery는 FDR DEG 검증이 아닙니다.
